In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# We will use dummy data matching the shape from your previous notebook
B, T, C = 1, 11, 16  # Batch Size, Sequence Length (Tokens), Embedding Dim
x = torch.randn(B, T, C)

# In a single head, we project the 16-dim embeddings into a smaller space
head_size = 16 
key_layer = nn.Linear(C, head_size, bias=False)
query_layer = nn.Linear(C, head_size, bias=False)
value_layer = nn.Linear(C, head_size, bias=False)

k = key_layer(x)   # Shape: (B, T, head_size)
q = query_layer(x) # Shape: (B, T, head_size)

# 1. SCALED DOT-PRODUCT: Tokens multiply their Queries against all other Keys
# This creates a (11x11) grid of "attention scores" or affinities
wei = q @ k.transpose(-2, -1) * (head_size ** -0.5) 

# 2. CAUSAL MASKING: We hide future tokens. Token 3 cannot read Token 4's data.
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))

# 3. SOFTMAX: Convert the raw scores into percentages/probabilities that sum to 1.0
wei = F.softmax(wei, dim=-1)

# 4. AGGREGATION: Multiply the weights by the actual Values
v = value_layer(x)
out = wei @ v

print("Attention weights (wei) shape:", wei.shape)
print("Attention weights for first token looking at all 11 tokens:\n", torch.round(wei[0, 0, :] * 100) / 100)
print("\nFinal single-head output shape:", out.shape)

Attention weights (wei) shape: torch.Size([1, 11, 11])
Attention weights for first token looking at all 11 tokens:
 tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], grad_fn=<DivBackward0>)

Final single-head output shape: torch.Size([1, 11, 16])


In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, n_embd):
        super().__init__()
        assert n_embd % num_heads == 0
        
        self.n_head = num_heads
        self.n_embd = n_embd
        
        # PyTorch optimization: compute Q,K,V for all heads at once using one large linear layer
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        
        # Output projection layer
        self.c_proj = nn.Linear(n_embd, n_embd)
        
        # Causal mask buffer
        self.register_buffer("bias", torch.tril(torch.ones(100, 100)).view(1, 1, 100, 100))

    def forward(self, x):
        B, T, C = x.size()
        
        # Compute Q, K, V
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        
        # Reshape to isolate the heads: (Batch, Num_Heads, Time, Head_Size)
        head_dim = C // self.n_head
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)
        
        # Compute attention scores
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        
        # Apply Causal Mask
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        
        # Multiply by values
        y = att @ v
        
        # Re-assemble all head outputs side by side back into the original shape
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        
        # Final linear projection
        out = self.c_proj(y)
        return out

# Initialize 4 heads, processing our 16-dimensional tokens
mha = MultiHeadAttention(num_heads=4, n_embd=16)
mha_output = mha(x)

print("Multi-Head Attention Output shape:", mha_output.shape)

Multi-Head Attention Output shape: torch.Size([1, 11, 16])
